# 03_pc_<agreement>_<source>_to_<target>
Run-all-safe enforcement pipeline notebook.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit.environment_config import bootstrap_fabric_env, get_path, load_fabric_config
from fabricops_kit.runtime_context import (
    assert_notebook_name_valid,
    build_runtime_context,
    generate_run_id,
    validate_notebook_name,
)
from fabricops_kit.fabric_input_output import lakehouse_table_read, lakehouse_table_write, warehouse_read, warehouse_write
from fabricops_kit.data_contracts import (
    extract_business_keys,
    extract_required_columns,
    get_executable_quality_rules,
    load_latest_approved_contract,
)
from fabricops_kit.data_quality import assert_dq_passed, run_dq_rules
from fabricops_kit.technical_audit_columns import add_audit_columns, add_datetime_features, add_hash_columns, default_technical_columns
from fabricops_kit.data_profiling import generate_metadata_profile, profile_dataframe_to_metadata
from fabricops_kit.data_product_metadata import build_dataset_run_record, build_quality_result_records
from fabricops_kit.data_lineage import build_lineage_from_notebook_code, build_lineage_handover_markdown, build_lineage_records


In [ ]:
ENV_NAME = "dev"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"
TARGET_KIND = "lakehouse"
SOURCE_TABLE = "TODO_source_table"
TARGET_TABLE = "TODO_target_table"
DATASET_NAME = "target_dataset"
WRITE_MODE = "overwrite"


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])
RUN_ID = generate_run_id(prefix="pc")
runtime_context = build_runtime_context(dataset_name=DATASET_NAME, environment=ENV_NAME, source_table=SOURCE_TABLE, target_table=TARGET_TABLE, run_id=RUN_ID)


## Load approved contract and fail fast
Load approved metadata-table contract immediately after source read and validate required columns before transformation steps.


In [ ]:
metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
contract = load_latest_approved_contract(
    metadata_path,
    dataset_name=DATASET_NAME,
    object_name=SOURCE_TABLE,
    contract_type="source_input",
)


In [ ]:
if SOURCE_KIND == "lakehouse":
    source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
required_columns = extract_required_columns(contract)
business_keys = extract_business_keys(contract)

missing = sorted(set(required_columns) - set(df_source.columns))
if missing:
    raise ValueError(f"Fail fast: missing required source columns: {missing}")


In [ ]:
# TODO: approved deterministic transformation logic only
df_transformed = df_source


In [ ]:
rules = DQ_RULES.get(TARGET_TABLE, [])


In [ ]:
DATETIME_COLUMN = None
DATETIME_PREFIX = None
if DATETIME_COLUMN:
    df_transformed = add_datetime_features(
        df_transformed,
        datetime_column=DATETIME_COLUMN,
        prefix=DATETIME_PREFIX,
    )


In [ ]:
BUSINESS_KEYS = list(business_keys or [])
df_standard = add_audit_columns(df_transformed, run_id=RUN_ID)
if BUSINESS_KEYS:
    df_standard = add_hash_columns(
        df_standard,
        business_keys=BUSINESS_KEYS,
    )


In [ ]:
tech_cols = default_technical_columns()
required_output_columns = set(extract_required_columns(contract) or [])
actual_output_cols = set(c for c in df_standard.columns if c not in tech_cols)
if required_output_columns and required_output_columns != actual_output_cols:
    raise ValueError("Fail fast: output contract mismatch.")


In [ ]:
# Human-approved contract and DQ rules are enforced here; AI does not decide pipeline outcomes.
# Run DQ checks on standardized output before writing target
dq_result = run_dq_rules(df_standard, table_name=TARGET_TABLE, rules=rules, fail_on_error=False)
display(dq_result)
dq_results_path = get_path(ENV_NAME, "metadata", config=CONFIG)
lakehouse_table_write(dq_result, dq_results_path, "DQ_RESULTS", mode="append")
assert_dq_passed(dq_result)

if TARGET_KIND == "lakehouse":
    target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
    lakehouse_table_write(df_standard, target_path, TARGET_TABLE, mode=WRITE_MODE)
else:
    warehouse_write(df_standard, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)


In [ ]:
if TARGET_KIND == "lakehouse":
    target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
    df_written = lakehouse_table_read(target_path, TARGET_TABLE)
else:
    df_written = warehouse_read(env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, config=CONFIG)
if df_written.count() == 0:
    raise ValueError("Fail fast: target write produced zero rows.")


In [ ]:
output_profile = generate_metadata_profile(df_written, dataset_name=DATASET_NAME)
output_profile_rows = profile_dataframe_to_metadata(df_written, dataset_name=DATASET_NAME)


In [ ]:
dq_result_records = [r.asDict(recursive=True) for r in dq_result.collect()]
quality_records = build_quality_result_records(
    {"results": dq_result_records, "can_continue": True},
    run_id=RUN_ID,
    dataset_name=DATASET_NAME,
    table_name=SOURCE_TABLE,
    table_stage="source",
)
dataset_record = build_dataset_run_record(
    run_id=RUN_ID,
    dataset_name=DATASET_NAME,
    environment=ENV_NAME,
    source_table=SOURCE_TABLE,
    target_table=TARGET_TABLE,
    status="completed",
    started_at_utc=runtime_context.get("started_at_utc"),
    row_count_source=df_source.count(),
    row_count_output=df_standard.count(),
)


## Lineage placeholder (deterministic scanning)

Deterministic lineage scanning requires notebook code text as input.

- Fabric notebook runtime does not reliably provide `__file__`.
- Use a placeholder code string for now, or paste collected notebook code text.
- This will be replaced in a follow-up PR when a Fabric notebook-cell collector helper is added.


In [ ]:
NOTEBOOK_CODE_FOR_LINEAGE = """
# Paste or collect notebook code here if deterministic lineage scanning is required.
# In a future Fabric helper, this can be replaced by a notebook-cell collector.
"""

lineage = build_lineage_from_notebook_code(
    NOTEBOOK_CODE_FOR_LINEAGE,
    use_ai=False,
)

lineage_records = build_lineage_records(
    dataset_name=DATASET_NAME,
    run_id=RUN_ID,
    source_tables=[SOURCE_TABLE],
    target_table=TARGET_TABLE,
    transformation_steps=lineage.get("steps", []),
)

display(lineage_records)
print(build_lineage_handover_markdown(lineage))
